Summer Algal Bloom Trend Analysis (2003–2020)
- Dataset: MODIS-Aqua L3SMI Chlorophyll-a
- Region: Any coastal region's shape file upload in your GEE account
- Season: Summer (March–May)
- Methods:
  * Bloom thresholding (Chl-a > 20 mg/m³)
  * Bloom area estimation
  * Mann–Kendall trend test
  * Sen’s slope estimation
  * Significance testing (p ≤ 0.05)

In [1]:
# 1. IMPORTS & EARTH ENGINE INITIALIZATION
import ee
import geemap
import numpy as np
import math
import matplotlib.pyplot as plt

ee.Authenticate()
ee.Initialize()
geemap.ee_initialize()

In [2]:
# 2. MAP VIEW
Map = geemap.Map(center=[10, 80], zoom=5)  #[latitude, longitude]
Map

Map(center=[10, 80], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tran…

In [3]:
# 3. STUDY REGION
# Any Ocean shapefile (GEE Asset)
northernio = ee.FeatureCollection("GEEassetpath")  #"projects/ee-name/assets/region"
ionorthern = northernio.geometry()

In [4]:
# 4. MODIS CHLOROPHYLL DATA (SUMMER)
dataset = (
    ee.ImageCollection("NASA/OCEANDATA/MODIS-Aqua/L3SMI")
    .select("chlor_a")
    .filter(ee.Filter.calendarRange(3, 5, "month"))     # March–May
    .filter(ee.Filter.calendarRange(2003, 2020, "year"))
)

def add_year(image):
    return image.set("year", image.date().get("year"))

dataset_summer = dataset.map(add_year)

In [5]:
# Clip to study region
chl_clipped = dataset_summer.map(lambda img: img.clip(ionorthern))

In [6]:
# 5. BLOOM MASK & AREA CALCULATION
BLOOM_THRESHOLD = 3.4  # mg/m³

def bloom_mask(image):
    mask = image.gt(BLOOM_THRESHOLD)
    return (
        image.updateMask(mask)
        .select("chlor_a")
        .copyProperties(image, ["system:time_start"])
    )

def bloom_area(image):
    mask = image.gt(BLOOM_THRESHOLD)
    area = mask.multiply(ee.Image.pixelArea()).divide(1e6)  # km²
    return (
        area.select("chlor_a")
        .copyProperties(image, ["system:time_start"])
    )

bloom_images = chl_clipped.map(bloom_mask)
bloom_area_images = chl_clipped.map(bloom_area)

In [7]:
# 6. YEARLY MAX BLOOM AREA (SUMMER)
start_year = 2003
end_year = 2020

years = ee.List.sequence(start_year, end_year)

def yearly_max(y):
    img = (
        bloom_area_images
        .filter(ee.Filter.calendarRange(y, y, "year"))
        .max()
    )
    return img.set(
        "year", y,
        "system:time_start", ee.Date.fromYMD(y, 1, 1).millis()
    )

yearly_bloom = ee.ImageCollection.fromImages(years.map(yearly_max))
yearly_mean = yearly_bloom.mean()

In [8]:
# Export mean bloom area
geemap.ee_export_image_to_drive(
    yearly_mean,
    description="Summer_AlgalBloom_Mean",
    folder="SeasonalTrend",
    region=ionorthern,
    scale=4000
)

In [9]:
# 7. MANN–KENDALL TREND ANALYSIS
after_filter = ee.Filter.lessThan(
    leftField="system:time_start",
    rightField="system:time_start"
)

joined = ee.ImageCollection(
    ee.Join.saveAll("after").apply(
        primary=yearly_bloom,
        secondary=yearly_bloom,
        condition=after_filter
    )
)

def mk_sign(i, j):
    return (
        ee.Image(j).neq(i)
        .multiply(ee.Image(j).subtract(i).clamp(-1, 1))
        .int()
    )

def mk_map(current):
    after = ee.ImageCollection.fromImages(current.get("after"))
    return after.map(
        lambda img: mk_sign(current, img).unmask(0).float()
    )

kendall = ee.ImageCollection(joined.map(mk_map).flatten())
kendall_stat = kendall.reduce("sum", 2)

In [10]:
# Spatial smoothing
kendall_smoothed = kendall_stat.reduceNeighborhood(
    reducer=ee.Reducer.mean(),
    kernel=ee.Kernel.circle(15)
)

geemap.ee_export_image_to_drive(
    kendall_smoothed,
    description="Summer_MannKendall_BloomArea",
    folder="Regression",
    region=ionorthern,
    scale=4000
)

In [11]:
# 8. SEN’S SLOPE
def sen_slope(i, j):
    return (
        ee.Image(j)
        .subtract(i)
        .divide(
            ee.Image(j).date().difference(
                ee.Image(i).date(), "days"
            )
        )
        .rename("slope")
        .float()
    )

def slope_map(current):
    after = ee.ImageCollection.fromImages(current.get("after"))
    return after.map(lambda img: sen_slope(current, img))

slopes = ee.ImageCollection(joined.map(slope_map).flatten())
sens_slope = slopes.reduce(ee.Reducer.median(), 2)

In [12]:
# 9. SIGNIFICANCE TESTING
count = joined.count()

def variance(n):
    return n.multiply(n.subtract(1)).multiply(
        n.multiply(2).add(5)
    )

kendall_variance = variance(count).divide(18).float()

zero = kendall_stat.multiply(kendall_stat.eq(0))
pos = kendall_stat.multiply(kendall_stat.gt(0)).subtract(1)
neg = kendall_stat.multiply(kendall_stat.lt(0)).add(1)

z = (
    zero
    .add(pos.divide(kendall_variance.sqrt()))
    .add(neg.divide(kendall_variance.sqrt()))
)

def cdf(z):
    return ee.Image(0.5).multiply(
        ee.Image(1).add(z.divide(ee.Image(2).sqrt()).erf())
    )

p_value = ee.Image(1).subtract(cdf(z.abs()))
significance_summer = p_value.lte(0.05)